In [ ]:
import numpy as np
import numpy.random as rand
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from pathlib import Path

# 경로 설정
print("💻 로컬 환경에서 실행")

# 현재 디렉토리에서 PDSI 폴더 찾기
current_dir = Path.cwd()

if 'PDSI' in str(current_dir):
    while current_dir.name != 'PDSI' and current_dir.parent != current_dir:
        current_dir = current_dir.parent
    PROJECT_ROOT = current_dir
else:
    # 기본 경로 사용
    PROJECT_ROOT = Path('/Users/mkim/Library/CloudStorage/GoogleDrive-wuj2293@gmail.com/My Drive/Research/PDSI')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = current_dir

# PDSI_data 폴더는 PDSI 프로젝트 밖에 생성
DATA_ROOT = PROJECT_ROOT.parent / 'PDSI_data'
WEIGHT_DIR = DATA_ROOT / 'weights'

# 디렉토리 생성
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📂 프로젝트 경로: {PROJECT_ROOT}")
print(f"📂 데이터 루트: {DATA_ROOT}")
print(f"📂 가중치 저장 경로: {WEIGHT_DIR}")

In [2]:
#rule 이진법함수
def get_wolfram_rule( rule_number ):
    '''
    Gets the mapping for the given rule.
    Each rule is essentially the binary version of the rule number
    '''
    binary_rep = str(bin(rule_number))[2:]
    # prepend 0s so that the are 8 bits
    binary_rep = [0]*(8-len(binary_rep)) + [int(b) for b in binary_rep]

    mapping = {
         (0,0,0): binary_rep[7],
         (0,0,1): binary_rep[6],
         (0,1,0): binary_rep[5],
         (0,1,1): binary_rep[4],
         (1,0,0): binary_rep[3],
         (1,0,1): binary_rep[2],
         (1,1,0): binary_rep[1],
         (1,1,1): binary_rep[0],}
    return mapping

In [3]:
#다음세대 계산함수
def get_next_seq_ped(previous_seq, rule, remain): #룰은 이진법 함수로 넣어야한다.
    '''
    Use a rule to generate the next sequence for an automoton
    '''
    seq_length = len(previous_seq) - 2
    rule_length = 3
    sub_seqs = [tuple(previous_seq[i:i + rule_length]) for i in range(0, seq_length)]
    new_seq = [rule[sub_seq] for sub_seq in sub_seqs]
    # pading the ends
    new_seq.insert(0, remain)
    new_seq.append(remain)
    return new_seq

In [4]:
# 400세대의 1의 개수를 구하는 코드
def counting_1(rule_num, init_num, seed_num):
    # 초기값에 따른 렌덤 행렬 생성
    rand.seed(seed_num)
    idx = rand.choice(400 ,init_num ,replace=False) # 400 크기 행렬
    prev = np.zeros(400) # 400 크기 행렬
    prev[idx] =1
    # 규칙에 따라 세대별 1개수 생성
    y = np.zeros(400) # 400세대 생성
    for a in range(400): # 400세대 생성
        y[a] = round(np.sum(prev))
        prev = get_next_seq_ped(prev, get_wolfram_rule(rule_num-round(rule_num % 2)), round(rule_num % 2))
    return y
    
# # 결과 그림    
# plt.plot(counting_1(31, 100, 101))
# plt.show()

In [ ]:
# 초기값마다 weight 값 구하는 코드
raw_data = np.zeros((256, 400), dtype=object)
for i in range(256):  # 모든 규칙
#even = [2 * x for x in range(128)]
#for i in even: # 짝수 규칙
    for j in range(400):
            y=np.zeros(400)
            for k in range(10):
                y+= counting_1(i,j,k)
            y= y/10
            y = y.reshape(-1, 1) # (400,)의 1D 배열에서 (400, 1) 2D 배열로 변환
            raw_data[i, j] = y
    print(i)
    np.save(WEIGHT_DIR / 'my_array2.npy', raw_data)

In [5]:
def regression(rule_num, init_num, when):
    x = np.arange(400)
    x = x.reshape(-1, 1)
    
    y=np.zeros(400)
    for g in range(10):
        y+= counting_1(rule_num,init_num, g)
    y= y/10
    y = y.reshape(-1, 1) # (400,)의 1D 배열에서 (400, 1) 2D 배열로 변환
    
    model = LinearRegression()
    model.fit(x, y)

    X_new = np.array([[t]])  # when에서의 예측
    y_predict = model.predict(X_new)

    return float(y_predict[0][0])

In [ ]:
loaded_array = np.load(WEIGHT_DIR / 'weight_init_even.npy', allow_pickle=True)

In [ ]:
# regression 코드
init = 60
reg_value = np.zeros((128, 400))
x = np.arange(400)
x = x.reshape(-1, 1)

for i in range(128):
    for j in range(400):
        y = loaded_array[i, j]
        model = LinearRegression()
        model.fit(x, y)
        y_predict = model.predict(np.array([[init]]))
        reg_value[i, j] = y_predict
    print(i)
np.save(WEIGHT_DIR / 'weight_init60_even.npy', reg_value)

In [9]:
# 그림 그리는 코드
# rule = 17
# plt.plot(loaded_array[rule,100])
# plt.plot(loaded_array[rule,200])
# plt.plot(loaded_array[rule,300])
# plt.ylim(0, 400)
# plt.show()